# 01. Clinical Notes & Claims Data Exploration
### Clinical Note-to-Billing Code Consistency Checker (FWA Triage)

In this notebook, we explore the primary dataset (MTSamples clinical transcriptions) and synthetic CMS DE-SynPUF claims, perform statistical profiling, and execute automated data-quality checks.

In [ ]:
import sys
from pathlib import Path

# Add project root src to path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from clinical_billing_audit.data.loaders import load_mtsamples, load_cms_claims, load_icd9_mapping
from clinical_billing_audit.data.cleaners import clean_clinical_notes, clean_specialty_labels, profile_dataset
from clinical_billing_audit.data.quality import DataQualityChecker

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Load MTSamples Clinical Notes

In [ ]:
notes_raw = load_mtsamples("../data/raw/mtsamples.csv")
print(f"Total notes loaded: {len(notes_raw):,}")
notes_raw.head(3)

## 2. Clean Notes & Specialties

In [ ]:
notes_clean = clean_clinical_notes(notes_raw, min_length=50)
notes_clean = clean_specialty_labels(notes_clean)
print(f"Cleaned notes count: {len(notes_clean):,}")
print(f"Unique medical specialties: {notes_clean['medical_specialty'].nunique()}")

## 3. Data Quality Audit & Profiling

In [ ]:
qc = DataQualityChecker()
report = qc.run(notes_clean, dataset_type="notes")
report_df = report.summary_df()
print("Data Quality Summary:")
report_df

## 4. Specialty Distribution & Note Lengths

In [ ]:
notes_clean['char_length'] = notes_clean['transcription'].str.len()
notes_clean['word_count'] = notes_clean['transcription'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 15 Specialties
top_specs = notes_clean['medical_specialty'].value_counts().head(15)
sns.barplot(x=top_specs.values, y=top_specs.index, ax=axes[0], palette="Blues_r")
axes[0].set_title("Top 15 Medical Specialties in Dataset", fontsize=13)
axes[0].set_xlabel("Number of Notes")

# Note Length Distribution
sns.histplot(notes_clean['word_count'], bins=40, kde=True, ax=axes[1], color="#2563eb")
axes[1].set_title("Note Word Count Distribution", fontsize=13)
axes[1].set_xlabel("Word Count")

plt.tight_layout()
plt.show()

## 5. Summary & Key Findings
- Notes represent diverse clinical encounter types.
- Word counts range from brief consultations to detailed operative reports.
- Data is ready for NLP entity extraction.